# Notebook Laboratorio 3 Bases de Datos Avanzadas - Neo4J

## 1. Librerías y dependencias

In [2]:
import pandas as pd
from neo4j import GraphDatabase

## 2. Conexión y creación de BD Neo4J

In [4]:
URI = "bolt://localhost:7687"
AUTH = ("neo4j", "password123")
driver = GraphDatabase.driver(URI, auth=AUTH)
session = driver.session()

## 3. Procesamiento y Poblamiento de la BD Neo4J

In [ ]:
df = pd.read_csv('normativas/normativas_clasificadas_IA.csv')
df.fillna("", inplace=True)

reglas_referencia = {
    "Resolución Exenta N° 176 de 2020": "Resolución 176 de 2020",
    "Resolución Exenta N° 76 de 2021": "Resolución 76 de 2021",
    "Resolución Exenta N° 79 de 2025": "Resolución 79 de 2025",
    "Resolución N° 59 de 2025": "Resolución 59 de 2025",
    "Circular N° 38 de 2025": "Circular 38 de 2025",
    "Articulo 68 del Código Tributario": "Articulo 68 del Código Tributario"
}

reglas_palabras = {
    "boleta": "Contiene 'boleta'",
    "comprobante electrónico": "Contiene 'comprobante electrónico'",
    "registro de compra": "Contiene 'registro de compra'",
    "registro de venta": "Contiene 'registro de venta'",
    "cumplimiento tributario": "Contiene 'cumplimiento tributario'",
    "inicio de actividades": "Contiene 'inicio de actividades'",
    "medios de pago electrónicos": "Contiene 'medios de pago electrónicos'",
    "pos": "Contiene 'POS'",
    "p.o.s": "Contiene 'P.O.S'",
    "puntos de venta": "Contiene 'puntos de venta'",
    "operadores y administradores": "Contiene 'operadores y administradores'",
    "comercio electrónico": "Contiene 'comercio electrónico'"
}

for index, row in df.iterrows():
    nombre = str(row['name'])
    desc = str(row['description'])
    fuente = str(row['fuente'])
    url = str(row['url'])
    tipo_doc = str(row['tipo_documento'])
    cuerpo = str(row['cuerpo'])
    relevancia = str(row['relevancia'])
    explicacion = str(row['explicacion'])
    
    texto_completo = f"{nombre} {desc} {cuerpo} {explicacion}"
    texto_lower = texto_completo.lower()
    reglas_activadas = []
    
    for ref, regla in reglas_referencia.items():
        if ref.lower() in texto_lower:
            reglas_activadas.append((regla, f"Se detectó la referencia: {ref}"))
            
    for palabra, regla in reglas_palabras.items():
        if palabra.lower() in texto_lower:
            reglas_activadas.append((regla, f"Se detectó la palabra clave: {palabra}"))
            
    labels = ["Normativa"]
    if "Circular" in tipo_doc: 
        labels.append("Circular")
    if "Resolución" in tipo_doc or "Resolucion" in tipo_doc: 
        labels.append("Resolucion")
    
    if relevancia == "Relevante": 
        labels.append("Relevante")
    else: 
        labels.append("NoRelevante")
    
    if relevancia == "Relevante" and len(reglas_activadas) > 0:
        labels.append("ExplicacionValida")
    elif relevancia == "Relevante" and len(reglas_activadas) == 0:
        labels.append("RequiereRevision")
        labels.append("ExplicacionDebil")
    elif relevancia == "NoRelevante" and len(reglas_activadas) > 0:
        labels.append("RequiereRevision")
        
    labels_str = ":".join(labels)
    
    query_base = f"""
    MERGE (agente:AgenteIANormativo {{nombre: 'Agente IA Normativo'}})
    MERGE (f:Fuente {{nombre: $fuente}})
    MERGE (n:{labels_str} {{nombre: $nombre}})
      SET n.descripcion = $desc, 
          n.url = $url, 
          n.cuerpo = $cuerpo
          
    MERGE (n)-[:EMITIDA_POR]->(f)
    MERGE (n)-[:CLASIFICADA_POR]->(agente)
    
    CREATE (exp:ExplicacionIA {{texto: $explicacion}})
    MERGE (n)-[:TIENE_EXPLICACION]->(exp)
    """
    session.run(query_base, fuente=fuente, nombre=nombre, desc=desc, url=url, cuerpo=cuerpo, explicacion=explicacion)
    
    for regla_nombre, evidencia in reglas_activadas:
        query_reglas = """
        MATCH (n:Normativa {nombre: $nombre})
        MERGE (r:ReglaDeNegocio {nombre: $regla_nombre})
        MERGE (n)-[:ACTIVA_REGLA]->(r)
        
        CREATE (ev:EvidenciaTextual {texto: $evidencia})
        MERGE (n)-[:RESPALDADA_POR]->(ev)
        """
        session.run(query_reglas, nombre=nombre, regla_nombre=regla_nombre, evidencia=evidencia)

## 4. Auditoria Simulada

## 5. Pruebas de Consistencia

## 6. Pruebas de Disponibilidad